# Session 04 — Color Filtering and Masks

**Goal:** turn a colour rule into a visible mask, clean that mask, and measure its largest region.

This notebook uses a supplied still image, so it runs in Google Colab without webcam permissions. You will edit just the HSV limits, then inspect what changes.


## 1. Import the tools

OpenCV changes and measures images. NumPy stores the image as numbers. Matplotlib shows the results.


In [ ]:
# OpenCV gives us tools for working with image pixels.
import cv2

# NumPy stores image pixels as rows and columns of numbers.
import numpy as np

# Matplotlib displays images inside Google Colab.
import matplotlib.pyplot as plt


## 2. Load the lesson image

The image is stored in this course repository. `imdecode` turns the downloaded bytes into an image. OpenCV normally uses BGR order, so we immediately convert it to RGB for display.


In [ ]:
# urlopen downloads the original lesson image from GitHub.
from urllib.request import urlopen

# This is the image used throughout the lesson.
IMAGE_URL = "https://raw.githubusercontent.com/qershyahya/iskytech-computer-vision-lab/main/session-04/assets/mask-lab-source.png"

# Download the image file as bytes.
image_bytes = urlopen(IMAGE_URL).read()

# Turn those bytes into an OpenCV image.
# OpenCV first reads colour images in BGR order.
image_bgr = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)

# Change BGR to RGB so Matplotlib shows the colours correctly.
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Make one empty space for the original image.
plt.figure(figsize=(8, 4.5))

# Draw the image in that space.
plt.imshow(image)

# Hide the number grid around the image.
plt.axis("off")

# Give the image a clear title.
plt.title("Original lesson image")


## 3. Choose the blue rule

HSV means **Hue, Saturation, Value**. Each pixel has three values. The two lines below set the lowest and highest HSV value we will accept as blue.

**Your edit:** change one number only, rerun the next cell, then explain one change in the mask.


In [ ]:
# Each list is [Hue, Saturation, Value].
# Hue runs from 0 to 179 in OpenCV.
# Saturation and Value run from 0 to 255.

# EDIT: the smallest HSV values that will count as blue.
LOWER_BLUE = np.array([95, 70, 55])

# EDIT: the largest HSV values that will count as blue.
UPPER_BLUE = np.array([125, 255, 255])


## 4. Make and inspect the raw mask

`inRange` tests every pixel. White means the pixel passed the blue rule; black means it did not.


In [ ]:
# Describe every pixel using Hue, Saturation, and Value.
hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

# Test every HSV pixel against the blue limits.
# White means "keep this pixel". Black means "ignore this pixel".
raw_mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE)

# Make a figure with two image spaces.
plt.figure(figsize=(10, 4))

# Show the original image on the left.
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

# Show the first black-and-white mask on the right.
plt.subplot(1, 2, 2)
plt.imshow(raw_mask, cmap="gray")
plt.title("Raw mask: white = kept")
plt.axis("off")


## 5. Clean the mask carefully

A small **opening** removes isolated white specks. A small **closing** fills tiny black holes. The 5×5 kernel is the size of the cleaning tool. Do not make it larger unless you can point to a visible problem it fixes.


In [ ]:
# Make a small 5 by 5 square cleaning tool.
# It is called a kernel because OpenCV slides it across the mask.
KERNEL = np.ones((5, 5), np.uint8)

# Opening removes isolated white specks.
clean_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, KERNEL)

# Closing fills tiny black holes inside the kept region.
clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, KERNEL)

# Make a figure with two image spaces.
plt.figure(figsize=(10, 4))

# Show the mask before cleaning.
plt.subplot(1, 2, 1)
plt.imshow(raw_mask, cmap="gray")
plt.title("Before cleanup")
plt.axis("off")

# Show the mask after cleaning.
plt.subplot(1, 2, 2)
plt.imshow(clean_mask, cmap="gray")
plt.title("After cleanup")
plt.axis("off")


## 6. Measure the largest connected region

A connected component is a group of touching white pixels. We choose the largest group, draw its box and centre, and report its area. This is a rule-based measurement, not recognition of an object's identity.


In [ ]:
# Find groups of touching white pixels in the cleaned mask.
count, labels, stats, centres = cv2.connectedComponentsWithStats(clean_mask)

# Make a copy so the original image stays unchanged.
result = image.copy()

# Label 0 is always the black background.
# If there is no label 1 or higher, no blue region passed the rule.
if count <= 1:
    print("No blue region passed the current rule. Adjust one HSV limit and try again.")
else:
    # Ignore the background and choose the region with the largest area.
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    # Read the chosen region's left edge and top edge.
    x = stats[biggest, cv2.CC_STAT_LEFT]
    y = stats[biggest, cv2.CC_STAT_TOP]

    # Read the chosen region's width, height, and area.
    width = stats[biggest, cv2.CC_STAT_WIDTH]
    height = stats[biggest, cv2.CC_STAT_HEIGHT]
    area = stats[biggest, cv2.CC_STAT_AREA]

    # Read and round the chosen region's centre point.
    cx = int(centres[biggest, 0])
    cy = int(centres[biggest, 1])

    # Draw a cyan box around the largest region.
    cv2.rectangle(result, (x, y), (x + width, y + height), (25, 205, 224), 4)

    # Draw an orange dot at the region's centre.
    cv2.circle(result, (cx, cy), 8, (246, 138, 69), -1)

    # Report the measurement in plain language.
    print(f"Largest blue region: area = {area} pixels, centre = ({cx}, {cy})")

# Show the measured result.
plt.figure(figsize=(8, 4.5))
plt.imshow(result)
plt.title("Largest connected blue region")
plt.axis("off")


## 7. Check that the baseline works

This short check confirms that the supplied lesson image produced a meaningful measured region. It is a test of the pipeline, not proof that every new camera scene will work.


In [ ]:
# The baseline image should contain at least one non-background region.
assert count > 1, "The baseline mask found no blue region."

# The blue tile should be large enough to be a useful target.
assert area > 2000, "The detected blue region is unexpectedly small."

# A clear message makes the successful check visible in Colab.
print("PASS: the HSV mask found and measured the blue tile.")


## Evidence card

Write down:

1. Your final `LOWER_BLUE` and `UPPER_BLUE` values.
2. One unwanted pixel group that the raw mask included or missed.
3. Whether cleanup improved the result, and why.
4. The largest component's area and centre.
5. One lighting condition you predict could make this rule fail.
